In [1]:
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk import sent_tokenize
from sklearn.naive_bayes import MultinomialNB

from sklearn.model_selection import train_test_split
import re
import  string
import gensim
from gensim.utils import simple_preprocess

In [2]:
def preprocess(t):
    t=re.sub(re.compile('<.*?>'),'',t)
    t=t.lower()
    list_=stopwords.words('english')
    t=t.translate(str.maketrans('', '', string.punctuation))
    new_t=''
    for i in t.split():
        if i not in list_:
            new_t+=i+' '
    t=new_t[:-1]
    return t

In [3]:
df=pd.read_csv('data_mr.csv').iloc[10000:15000]
df.drop_duplicates(inplace=True)


In [4]:
df['text']=df['text'].apply(lambda x: preprocess(x))
    

In [5]:
doc=[]
for i in df['text']:
    raw=sent_tokenize(i)
    for j in raw:
        doc.append(simple_preprocess(j))
        

In [6]:
model=gensim.models.Word2Vec(window=5,min_count=2,vector_size=300)

In [7]:
model.build_vocab(doc)

In [8]:
model.train(doc,total_examples=model.corpus_count,epochs=model.epochs)

(2669355, 2910290)

In [9]:
def doc_vec(x):
    j=[i for i in x.split() if i in model.wv.index_to_key]
    return np.mean(model.wv[j],axis=0)

In [10]:
X=[]
for k in df['text']:
    X.append(doc_vec(k))
X=np.array(X)
    

In [11]:
y=df['label']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=7)

In [13]:
r=MultinomialNB()

In [14]:
r.fit(X_train,y_train)

ValueError: Negative values in data passed to MultinomialNB (input X).

In [ ]:
y_pred=r.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score,f1_score

In [ ]:
print(f1_score(y_test,y_pred))

In [ ]:
input_review1='this movie was the worst, i really hate it such a boring and ugly movie'
input_review2='this movie was beautiful and awesome i loved it'
t1=preprocess(input_review1)
t2=preprocess(input_review2)
Word2vec1=doc_vec(t1)
Word2vec2=doc_vec(t2)
print(Word2vec1)


In [ ]:
print(Word2vec2)

In [ ]:
neg_review = "this movie was the worst, i really hate it such a boring and ugly movie"
pos_review = "this movie was beautiful and awesome i really loved it from the bottom of my heart"

for review in [neg_review, pos_review]:
    vec = doc_vec(preprocess(review)).reshape(1,-1)

    print("Prediction:", r.predict(vec))
    print("Probabilities:", r.predict_proba(vec))
    print()

In [ ]:
reviews = [
    "excellent masterpiece amazing wonderful",
    "fantastic brilliant loved every second",
    "terrible horrible worst movie ever",
    "awful boring disgusting waste of time"
]

for review in reviews:
    vec = doc_vec(preprocess(review)).reshape(1,-1)
    print(review)
    print(r.predict(vec))
    print(r.predict_proba(vec))
    print()

In [ ]:
print(X_train.min())